# Analysis of ODP geospatial names - tree, tree-preservation-zone, listed-building-outline
**Author**:  Sian Teesdale <br>
**Date**:  10th August 2026 <br>
**Dataset Scope**: `tree`, `tree-preservation-zone`, `listed-building-outline` <br>
**Report Type**: Ad-hoc analysis <br>

## Purpose

Follows on from the broader [initial_odp_name_analysis.ipynb](initial_odp_name_analysis.ipynb) exploration. That notebook flagged blank names and bare reference-code-style names as suspicious across all five ODP geography datasets - but for `tree` and `tree-preservation-zone`, that's too blunt a check.

**`tree` / `tree-preservation-zone`**: the ODP guidance for these two datasets says the `name` field can legitimately be **any** of:
- a descriptive name
- the same value as the `reference` field (e.g. reference `TP1` -> name `TP1`)
- an address (the `address`/`address-text` field)
- blank, if the TPO/tree doesn't have a name

So a bare code like `T1` isn't necessarily a data quality problem - it might just be the reference being reused as the name, which is allowed. This notebook checks how much of the "weirdness" flagged previously is actually explained by these allowed patterns (also checking a fifth, informally-observed pattern - the name matching the `tree-preservation-order` reference), versus how much is still unexplained.

**`listed-building-outline` (LBO)**: LBO is the geospatial counterpart of the non-spatial `listed-building` (LB) dataset, linked via LBO's `listed-building` column against LB's `reference` column. The check here is different in kind: rather than asking whether an LBO name matches some *allowed* pattern, we ask whether it matches its *linked LB record's own name* - the expectation being that the two should agree, since they describe the same listed building. This surfaced two useful findings:
- Of LBO rows that link to an LB record, only ~51% have an exact name match - though many of the "mismatches" turn out to be LBO appending extra locating detail onto the LB name (e.g. LB `"NORTH LODGE"` -> LBO `"NORTH LODGE - B1318 (EAST SIDE) GOSFORTH PARK"`), not a real error.
- LBO rows with placeholder text instead of a name (`"No given name"`, `"No name for this Entry"`, `"No address supplied"`) are each a single-organisation habit rather than a general pattern. Critically, 844 of 845 `"No name for this Entry"` rows link straight to an LB record that already has a proper name - so the placeholder is masking a real name that's one join away, not describing a genuinely unnamed building. The other two placeholder phrases barely link to LB at all, suggesting their `listed-building` reference is stale/wrong rather than the building being unnamed. None of the three phrases appear anywhere in LB's own `name` field, confirming this is purely an LBO-side data entry convention.


In [1]:
import pandas as pd
import geopandas as gpd
import numpy as np
import os
import re
import urllib
from collections import Counter
from datetime import datetime

td = datetime.today().strftime('%Y-%m-%d')

pd.set_option("display.max_rows", 100)

data_dir = "../../data/db_downloads/"
os.makedirs(data_dir, exist_ok=True)


## Data Import

In [2]:
DATASET_SLUGS = [
    "listed-building-outline",
    "tree-preservation-zone",
    "tree",
]

dfs = {
    slug: pd.read_csv(f"https://files.planning.data.gov.uk/dataset/{slug}.csv")
    for slug in DATASET_SLUGS
}

{slug: df.shape for slug, df in dfs.items()}


/var/folders/ff/zpylthrn3877kx4h8lxvsvjm0000gn/T/ipykernel_7466/2132323433.py:8: DtypeWarning: Columns (14,15,19) have mixed types. Specify dtype option on import or set low_memory=False.
  slug: pd.read_csv(f"https://files.planning.data.gov.uk/dataset/{slug}.csv")
/var/folders/ff/zpylthrn3877kx4h8lxvsvjm0000gn/T/ipykernel_7466/2132323433.py:8: DtypeWarning: Columns (15,21) have mixed types. Specify dtype option on import or set low_memory=False.
  slug: pd.read_csv(f"https://files.planning.data.gov.uk/dataset/{slug}.csv")
/var/folders/ff/zpylthrn3877kx4h8lxvsvjm0000gn/T/ipykernel_7466/2132323433.py:8: DtypeWarning: Columns (14,16,17,22,24) have mixed types. Specify dtype option on import or set low_memory=False.
  slug: pd.read_csv(f"https://files.planning.data.gov.uk/dataset/{slug}.csv")


{'listed-building-outline': (126247, 22),
 'tree-preservation-zone': (102574, 22),
 'tree': (262304, 25)}

## Analysis

### `name` column patterns

Reusable checks: exact-duplicate names, common repeated phrases (e.g. boilerplate text like "Town and Country"), names that look like bare reference codes rather than descriptions, blank names, and - new in this notebook - whether a name is actually explained by matching the `reference` field or an address field (both allowed by the ODP guidance for `tree`/`tree-preservation-zone`).

In [22]:
def word_ngrams(text, n):
    words = re.findall(r"[a-z0-9\']+", text.lower())
    return {' '.join(words[i:i + n]) for i in range(len(words) - n + 1)}


def duplicate_names(df, name_col='name'):
    """Names reused across more than one row."""
    counts = df[name_col].value_counts()
    return counts[counts > 1]


def top_ngrams(df, name_col='name', n=2, min_count=5, top=15):
    """Most common n-word phrases, counted by number of distinct rows containing them."""
    names = df[name_col].dropna().astype(str)
    counter = Counter()
    for text in names:
        counter.update(word_ngrams(text, n))
    return [(phrase, count) for phrase, count in counter.most_common(top) if count >= min_count]


def code_like_names(df, name_col='name'):
    """Names that look like a bare reference code (e.g. '0164B2', '11/222', '16.018') rather than a description.
    Requires at least one digit so plain single-word place names (e.g. 'Napsbury') aren't misflagged."""
    return df[df[name_col].str.match(r'^(?=.*[0-9])[A-Za-z0-9./-]{1,20}$', na=False)]


def missing_names(df, name_col='name'):
    """Rows with a blank/missing name - covers both NaN cells and empty or whitespace-only strings."""
    return df[df[name_col].isna() | (df[name_col].astype(str).str.strip() == '')]


def _normalised(series):
    return series.astype(str).str.strip().str.lower()


def name_matches_reference(df, name_col='name', ref_col='reference'):
    """Rows where name is identical to reference - allowed per ODP guidance for tree/tree-preservation-zone."""
    name, ref = _normalised(df[name_col]), _normalised(df[ref_col])
    return df[df[name_col].notna() & df[ref_col].notna() & (name == ref)]


def name_matches_address(df, name_col='name', address_cols=('address', 'address-text')):
    """Rows where name is identical to an address field - allowed per ODP guidance for tree/tree-preservation-zone."""
    name = _normalised(df[name_col])
    mask = pd.Series(False, index=df.index)
    for col in address_cols:
        if col in df.columns:
            addr = _normalised(df[col])
            mask |= df[name_col].notna() & df[col].notna() & (name == addr)
    return df[mask]


def name_matches_tpo(df, name_col='name', tpo_col='tree-preservation-order'):
    """Rows where name is identical to the tree-preservation-order reference (e.g. 'TPO/2010/001', '10/1997').
    Not one of the ODP guidance's four listed allowed patterns, but observed in practice - LPAs reusing
    the TPO reference itself as the name, distinct from the dataset's own 'reference' field."""
    if tpo_col not in df.columns:
        return df.iloc[0:0]
    name, tpo = _normalised(df[name_col]), _normalised(df[tpo_col])
    return df[df[name_col].notna() & df[tpo_col].notna() & (name == tpo)]


def classify_name(df, name_col='name', ref_col='reference', address_cols=('address', 'address-text'), tpo_col='tree-preservation-order'):
    """Classify each row's name against the ODP guidance for tree/tree-preservation-zone:
    'blank', 'matches reference', 'matches address', 'matches tree-preservation-order', or 'description' (free text)."""
    result = pd.Series('description', index=df.index)

    blank_mask = df[name_col].isna() | (df[name_col].astype(str).str.strip() == '')
    result[blank_mask] = 'blank'

    name = _normalised(df[name_col])
    if ref_col in df.columns:
        ref = _normalised(df[ref_col])
        ref_mask = ~blank_mask & df[ref_col].notna() & (name == ref)
        result[ref_mask] = 'matches reference'

    for col in address_cols:
        if col in df.columns:
            addr = _normalised(df[col])
            addr_mask = ~blank_mask & (result == 'description') & df[col].notna() & (name == addr)
            result[addr_mask] = f'matches {col}'

    if tpo_col in df.columns:
        tpo = _normalised(df[tpo_col])
        tpo_mask = ~blank_mask & (result == 'description') & df[tpo_col].notna() & (name == tpo)
        result[tpo_mask] = f'matches {tpo_col}'

    return result


def unexplained_code_like_names(df, name_col='name', ref_col='reference', address_cols=('address', 'address-text'), tpo_col='tree-preservation-order'):
    """Code-like names that are NOT explained by matching reference, an address field, or the TPO reference.
    These are the ones still worth questioning - a bare code that doesn't correspond to any known field."""
    codes = code_like_names(df, name_col)
    explained = pd.concat([
        name_matches_reference(codes, name_col, ref_col),
        name_matches_address(codes, name_col, address_cols),
        name_matches_tpo(codes, name_col, tpo_col),
    ]).index.unique()
    return codes.drop(index=explained, errors='ignore')


def link_lbo_to_lb(lbo_df, lb_df, lbo_key='listed-building', lb_key='reference'):
    """Join listed-building-outline rows to their parent listed-building entity via
    lbo['listed-building'] == lb['reference']. Left join - rows with no reference set, or no
    matching listed-building row, keep NaN in the lb_name/lb_entity columns."""
    lbo_key_series = lbo_df[lbo_key].astype(str).str.strip()
    lb_lookup = lb_df[[lb_key, 'name', 'entity']].copy()
    lb_lookup['_lb_key'] = lb_lookup[lb_key].astype(str).str.strip()
    lb_lookup = lb_lookup.rename(columns={'name': 'lb_name', 'entity': 'lb_entity'})

    merged = lbo_df.assign(_lbo_key=lbo_key_series).merge(
        lb_lookup[['_lb_key', 'lb_name', 'lb_entity']],
        left_on='_lbo_key', right_on='_lb_key', how='left'
    )
    return merged.drop(columns=['_lbo_key', '_lb_key'])


def lbo_name_matches_lb(linked_df, lbo_name_col='name', lb_name_col='lb_name'):
    """Rows where the LBO's name is identical to its linked LB's name - the expected pattern."""
    lbo_name = _normalised(linked_df[lbo_name_col])
    lb_name = _normalised(linked_df[lb_name_col])
    return linked_df[linked_df[lb_name_col].notna() & (lbo_name == lb_name)]


### Per-dataset summary

In [12]:
for slug, df in dfs.items():
    n = len(df)
    dupes = duplicate_names(df)
    codes = code_like_names(df)
    blanks = missing_names(df)
    print(f"=== {slug} ({n} rows) ===")
    print(f"  {len(dupes)} distinct names reused across {dupes.sum()} rows ({dupes.sum() / n:.1%})")
    print(f"  {len(codes)} rows ({len(codes) / n:.1%}) look like bare reference codes")
    print(f"  {len(blanks)} rows ({len(blanks) / n:.1%}) have a blank/missing name")
    for gram_n in (2, 3):
        phrases = top_ngrams(df, n=gram_n)
        if phrases:
            top_phrase, top_count = phrases[0]
            print(f"  top {gram_n}-word phrase: '{top_phrase}' in {top_count} rows ({top_count / n:.1%})")
    print()


=== listed-building-outline (126247 rows) ===
  7062 distinct names reused across 25167 rows (19.9%)
  144 rows (0.1%) look like bare reference codes
  2245 rows (1.8%) have a blank/missing name
  top 2-word phrase: 'church of' in 5881 rows (4.7%)
  top 3-word phrase: 'church of st' in 4766 rows (3.8%)

=== tree-preservation-zone (102574 rows) ===
  7555 distinct names reused across 62984 rows (61.4%)
  12050 rows (11.7%) look like bare reference codes
  24291 rows (23.7%) have a blank/missing name
  top 2-word phrase: 'tree preservation' in 18789 rows (18.3%)
  top 3-word phrase: 'tree preservation order' in 17671 rows (17.2%)

=== tree (262304 rows) ===
  16721 distinct names reused across 185088 rows (70.6%)
  43535 rows (16.6%) look like bare reference codes
  50180 rows (19.1%) have a blank/missing name
  top 2-word phrase: 'tree preservation' in 41153 rows (15.7%)
  top 3-word phrase: 'tree preservation order' in 31492 rows (12.0%)



---

### Name field guidance compliance (`tree` & `tree-preservation-zone`)

Classifies every name into one of: `blank`, `matches reference`, `matches address`/`address-text`, `matches tree-preservation-order`, or `description` (free text). The first four are all reasonable to treat as "not a problem" - `matches reference` and `matches address(-text)` are explicitly allowed by the guidance, `blank` is allowed if the TPO/tree has no name, and `matches tree-preservation-order` isn't in the guidance's four bullet points but is a very similar pattern (the LPA's own TPO reference reused as the name). Only `description` needs eyeballing for boilerplate/placeholder issues (already covered above), and even that's fine if it's a genuine descriptive name.

In [5]:
for slug in ["tree-preservation-zone", "tree"]:
    df = dfs[slug]
    classes = classify_name(df)
    print(f"=== {slug} ({len(df)} rows) ===")
    print((classes.value_counts() / len(df)).apply(lambda x: f"{x:.1%}").to_string())
    print()


=== tree-preservation-zone (102574 rows) ===
description             73.8%
blank                   23.7%
matches address-text     1.3%
matches reference        1.2%

=== tree (262304 rows) ===
description             75.0%
blank                   19.1%
matches address-text     3.0%
matches reference        2.9%



### Unexplained code-like names

Of the names that look like bare codes, how many are actually just the `reference` or an address (allowed), versus a bare code that matches neither (still worth questioning)?

In [6]:
for slug in ["tree-preservation-zone", "tree"]:
    df = dfs[slug]
    codes = code_like_names(df)
    unexplained = unexplained_code_like_names(df)
    print(f"=== {slug} ===")
    print(f"  {len(codes)} code-like names total")
    print(f"  {len(codes) - len(unexplained)} explained by reference/address (allowed)")
    print(f"  {len(unexplained)} unexplained - bare code matching neither reference nor address")
    print()


=== tree-preservation-zone ===
  12050 code-like names total
  965 explained by reference/address (allowed)
  11085 unexplained - bare code matching neither reference nor address

=== tree ===
  43535 code-like names total
  3280 explained by reference/address (allowed)
  40255 unexplained - bare code matching neither reference nor address



In [18]:
# Inspect the unexplained code-like names for one dataset
dataset = "tree-preservation-zone"  # tree-preservation-zone, tree

df = dfs[dataset]
unexplained = unexplained_code_like_names(df)
print(f"{len(unexplained)} unexplained code-like names in '{dataset}'")
unexplained[['entity', 'organisation-entity', 'name', 'reference']]


5936 unexplained code-like names in 'tree-preservation-zone'


,entity,organisation-entity,name,reference
9444,19113163,152,W1,W1 No.3 1981
9445,19113164,152,W1,W1 No.5 1982
9446,19113165,152,W1,W1 No.2 1986
9447,19113166,152,W2,W2 No.3 1986
9448,19113167,152,W1,W1 No.3 1986
...,...,...,...,...
61249,19173065,195,G1,TPO2025_127_G_001
61250,19173066,195,G2,TPO2025_130_G_002
61251,19173067,195,G3,TPO2025_130_G_003
61252,19173068,195,G1,TPO2025_130_G_001


In [20]:
# Inspect example rows for any name classification (reuses `dataset`/`df` set above)
category = "matches tree-preservation-order"  # blank, matches reference, matches address-text, matches tree-preservation-order, description

classes = classify_name(df)
examples = df[classes == category]

# Sanity check: classify_name checks 'matches reference' before 'matches tree-preservation-order',
# so a row where name == reference == tpo is already claimed by 'matches reference' and can't appear
# here too - but assert it explicitly rather than relying on that ordering being remembered.
also_matches_reference = _normalised(examples['name']) == _normalised(examples['reference'])
assert not also_matches_reference.any(), "found a 'matches tree-preservation-order' row that also matches reference - check classify_name precedence"

print(f"{len(examples)} rows in '{dataset}' classified as '{category}' (none also match reference)")
examples[['entity', 'organisation-entity', 'name', 'reference', 'tree-preservation-order']].head(20)


5149 rows in 'tree-preservation-zone' classified as 'matches tree-preservation-order' (none also match reference)


,entity,organisation-entity,name,reference,tree-preservation-order
51830,19163646,205,1/1997,TPO01,1/1997
51831,19163647,205,1/1997,TPO02,1/1997
51832,19163648,205,1/1997,TPO03,1/1997
51833,19163649,205,12/1985,TPO04,12/1985
51834,19163650,205,41/1981,TPO05,41/1981
51835,19163651,205,41/1981,TPO06,41/1981
51836,19163652,205,24/1995,TPO07,24/1995
51837,19163653,205,22/1995,TPO08,22/1995
51838,19163654,205,22/1995,TPO09,22/1995
51839,19163655,205,22/1995,TPO10,22/1995


---

### Drill down into a dataset

Swap `dataset` below to inspect a specific dataset's full n-gram breakdown, then a chosen `phrase` to see the matching rows.

#### Listed building outline

In [ ]:
dataset = "listed-building-outline" # listed-building-outline, tree-preservation-zone, tree

df = dfs[dataset]
names = df['name'].dropna().astype(str)
for gram_n in range(1, 5):
    print(f"--- Most common {gram_n}-word phrases in '{dataset}' ---")
    for phrase, count in top_ngrams(df, n=gram_n, min_count=5, top=15):
        print(f"  {count:5d}  ({count / len(names):.1%})  {phrase}")
    print()


In [ ]:
# Code-like names for the chosen dataset (reuses `dataset`/`df` set above)
codes = code_like_names(df)
print(f"{len(codes)} rows ({len(codes) / len(df):.1%}) look like bare reference codes in '{dataset}'")
codes[['entity', 'organisation-entity', 'name']]

In [ ]:
# Blank/missing names for the chosen dataset (reuses `dataset`/`df` set above)
blanks = missing_names(df)
print(f"{len(blanks)} rows ({len(blanks) / len(df):.1%}) have a blank/missing name in '{dataset}'")
blanks[['entity', 'organisation-entity', 'name']]

In [ ]:
# Swap in whatever phrase looked suspicious above
phrase = "No given name"

matches = df[df['name'].str.contains(phrase, case=False, na=False)]
print(f"{len(matches)} rows in '{dataset}' contain '{phrase}'")
matches[['entity', 'organisation-entity', 'name']].head(20)


In [ ]:
# Swap in whatever phrase looked suspicious above
phrase = "No name for this Entry"

matches = df[df['name'].str.contains(phrase, case=False, na=False)]
print(f"{len(matches)} rows in '{dataset}' contain '{phrase}'")
matches[['entity', 'organisation-entity', 'name']].head(20)


In [ ]:
# Swap in whatever phrase looked suspicious above
phrase = "No address supplied"

matches = df[df['name'].str.contains(phrase, case=False, na=False)]
print(f"{len(matches)} rows in '{dataset}' contain '{phrase}'")
matches[['entity', 'organisation-entity', 'name']].head(20)


##### Cross-check against the `listed-building` dataset

`listed-building-outline` (LBO) is the geospatial counterpart of `listed-building` (LB) - they're linked via LBO's `listed-building` column, which should hold the value of LB's `reference` column. The check here: does the LBO's `name` match the name of the LB it's linked to? If not, one of the two datasets has drifted, or the LBO is describing a specific part/location of the listed building rather than just repeating its name.

In [23]:
lb = pd.read_csv("https://files.planning.data.gov.uk/dataset/listed-building.csv")

lbo = dfs["listed-building-outline"]
linked = link_lbo_to_lb(lbo, lb)

n = len(linked)
has_ref = linked["listed-building"].notna()
has_link = linked["lb_name"].notna()
matches = lbo_name_matches_lb(linked)

print(f"{n} total listed-building-outline rows")
print(f"  {has_ref.sum()} ({has_ref.mean():.1%}) have a 'listed-building' reference set")
print(f"  {has_link.sum()} ({has_link.sum() / n:.1%}) successfully link to a listed-building row")
print(f"  {len(matches)} ({len(matches) / has_link.sum():.1%} of linked rows) have LBO name == LB name exactly")


126247 total listed-building-outline rows
  112421 (89.0%) have a 'listed-building' reference set
  103189 (81.7%) successfully link to a listed-building row
  52486 (50.9% of linked rows) have LBO name == LB name exactly


In [27]:
mismatches.loc[mismatches['name'].str.contains('THORNTREE', na=False)]

,dataset,end-date,entity,entry-date,geojson,geometry,name,organisation-entity,point,prefix,...,address-text,description,listed-building,notes,organisation,uprns,wikidata,wikipedia,lb_name,lb_entity
1792,listed-building-outline,NaN,42112288,1987-03-30,NaN,"MULTIPOLYGON (((-1.685500 54.982691,-1.685481 ...",THORNTREE FARMHOUSE - WEST ROAD (SOUTH SIDE),228,POINT (-1.685615 54.982703),listed-building-outline,...,NaN,NaN,1024726,Thorntree Farmhouse,NaN,NaN,NaN,NaN,THORNTREE FARMHOUSE,31482538.0


In [24]:
# Linked rows where the LBO name and LB name differ
mismatches = linked[has_link & ~linked.index.isin(matches.index)]
print(f"{len(mismatches)} linked rows where LBO name != LB name")
mismatches[['entity', 'organisation-entity', 'name', 'lb_name', 'listed-building']].head(20)


50703 linked rows where LBO name != LB name


,entity,organisation-entity,name,lb_name,listed-building
1778,42112274,228,"WEST GATEWAY, WALLS AND GATES OF ST. NICHOLAS'...","WEST GATEWAY, WALLS AND GATES OF ST NICHOLAS C...",1024710
1779,42112275,228,HALL OF CHURCH OF ST. JAMES AND ST. BASIL - WI...,HALL OF CHURCH OF ST JAMES AND ST BASIL,1024711
1780,42112276,228,NORTH LODGE - B1318 (EAST SIDE) GOSFORTH PARK,NORTH LODGE,1024712
1781,42112277,228,"ICE HOUSE, GOSFORTH PARK",ICE HOUSE ABOUT 150 METRES EAST OF NORTH LODGE,1024713
1782,42112278,228,ENTRANCE LODGE TO BRANDLING HOUSE - B1318 (EAS...,ENTRANCE LODGE TO BRANDLING HOUSE,1024714
1783,42112279,228,BORDER MINSTREL PUBLIC HOUSE TO NORTH WEST OF ...,BORDER MINSTREL PUBLIC HOUSE TO NORTH WEST OF ...,1024715
1784,42112280,228,GATE PIERS AND STABLES NORTH OF BRANDLING HOUS...,GATE PIERS AND STABLES NORTH OF BRANDLING HOUSE,1024716
1785,42112281,228,GAS LAMPS AT ENTRANCE TO SOUTH DRIVE - A696(T)...,GAS LAMPS AT ENTRANCE TO SOUTH DRIVE,1024717
1786,42112282,228,"STABLES AND COACHHOUSE, NORTH OF WOOLSINGTON H...","STABLES AND COACHHOUSE, NORTH OF WOOLSINGTON HALL",1024718
1787,42112283,228,BULLOCK STEADS FARMHOUSE - PONTELAND ROAD (NOR...,BULLOCK STEADS FARMHOUSE,1024719


In [28]:
# Of the mismatches, how many have a bare code-like LBO name (e.g. just a house number)?
code_mismatches = code_like_names(mismatches)
print(f"{len(code_mismatches)} of {len(mismatches)} mismatches ({len(code_mismatches) / len(mismatches):.1%}) have a bare code-like LBO name")
code_mismatches[['entity', 'organisation-entity', 'name', 'lb_name']].head(20)


72 of 50703 mismatches (0.1%) have a bare code-like LBO name


,entity,organisation-entity,name,lb_name
4379,42115994,90,No.8,"8, GOLDEN YARD"
4888,42116503,90,56,"56, GOODGE STREET"
6088,42117703,90,51,PINEAPPLE PUBLIC HOUSE
7151,42118766,111,77,"77, CHURCH HILL (See details for further addre..."
7174,42118789,111,1,"1, MAISON DIEU ROAD (See details for further a..."
7213,42118828,111,32,"32, HIGH STREET"
7214,42118829,111,67,"67, HIGH STREET"
7215,42118830,111,69,"69, HIGH STREET"
7216,42118831,111,90,"90, 91 AND 92, HIGH STREET"
7233,42118848,111,47,"47, STRAND STREET"


##### Do the placeholder-name rows link to a real LB record?

Earlier we found LBO rows with placeholder text instead of a name ("No given name", "No name for this Entry", "No address supplied"). If those rows link to an LB record that has a proper name, the placeholder is unnecessary - the real name is sitting right there in the linked dataset.

In [31]:
placeholder_phrases = ["No given name", "No name for this Entry", "No address supplied"]

for phrase in placeholder_phrases:
    subset = linked[linked['name'].str.lower() == phrase.lower()]
    has_ref = subset['listed-building'].notna()
    has_link = subset['lb_name'].notna()
    print(f"=== '{phrase}' ({len(subset)} rows) ===")
    print(f"  {has_ref.sum()} ({has_ref.mean():.1%}) have a 'listed-building' reference set")
    print(f"  {has_link.sum()} ({has_link.sum() / len(subset):.1%}) successfully link to an LB row with a real name")
    print(f"  organisation-entity breakdown:")
    print(subset['organisation-entity'].value_counts().to_string())
    print()


=== 'No given name' (220 rows) ===
  220 (100.0%) have a 'listed-building' reference set
  2 (0.9%) successfully link to an LB row with a real name
  organisation-entity breakdown:
organisation-entity
329    220

=== 'No name for this Entry' (845 rows) ===
  845 (100.0%) have a 'listed-building' reference set
  844 (99.9%) successfully link to an LB row with a real name
  organisation-entity breakdown:
organisation-entity
123    448
330    307
382     67
26      23

=== 'No address supplied' (149 rows) ===
  149 (100.0%) have a 'listed-building' reference set
  1 (0.7%) successfully link to an LB row with a real name
  organisation-entity breakdown:
organisation-entity
192    149



In [ ]:
# Do any of these placeholder phrases also show up in the LB dataset's own name field?
for phrase in placeholder_phrases:
    exact = lb[lb['name'].str.lower() == phrase.lower()]
    contains = lb[lb['name'].str.contains(phrase, case=False, na=False)]
    print(f"'{phrase}': {len(exact)} exact matches, {len(contains)} substring matches in LB")


In [30]:
# Inspect the linked examples for one placeholder phrase - shows the real LB name that could replace the placeholder
phrase = "No name for this Entry"  # No given name, No name for this Entry, No address supplied

subset = linked[linked['name'].str.lower() == phrase.lower()]
linked_examples = subset[subset['lb_name'].notna()]
print(f"{len(linked_examples)} of {len(subset)} '{phrase}' rows link to an LB record")
linked_examples[['entity', 'organisation-entity', 'name', 'lb_name', 'listed-building']].head(20)


844 of 845 'No name for this Entry' rows link to an LB record


,entity,organisation-entity,name,lb_name,listed-building
61289,42177151,123,No name for this Entry,"33, 37 AND 39, BAYFORD GREEN",1176582
61303,42177165,123,No name for this Entry,"26 AND 28, ASHENDENE ROAD",1101705
61306,42177168,123,No name for this Entry,"6 AND 8, CHURCH ROAD",1040070
61324,42177186,123,No name for this Entry,"3, ASHENDENE ROAD",1347827
61354,42177216,123,No name for this Entry,"31 AND 33, WORMLEY WEST END",1177242
61357,42177219,123,No name for this Entry,"20, MORGANS ROAD",1268837
61358,42177220,123,No name for this Entry,"78-84, HORNS MILL ROAD",1268861
61384,42177246,123,No name for this Entry,"76-79, COLD CHRISTMAS LANE",1078715
61450,42177312,123,No name for this Entry,"43, HIGH ROAD",1347495
61457,42177319,123,No name for this Entry,"23, BALDOCK STREET",1217402


#### Tree Preservation Zone

In [8]:
dataset = "tree-preservation-zone" # listed-building-outline, tree-preservation-zone, tree

df = dfs[dataset]
names = df['name'].dropna().astype(str)
for gram_n in range(1, 5):
    print(f"--- Most common {gram_n}-word phrases in '{dataset}' ---")
    for phrase, count in top_ngrams(df, n=gram_n, min_count=5, top=15):
        print(f"  {count:5d}  ({count / len(names):.1%})  {phrase}")
    print()


--- Most common 1-word phrases in 'tree-preservation-zone' ---
  21895  (28.0%)  the
  19080  (24.4%)  tree
  18866  (24.1%)  preservation
  18809  (24.0%)  order
  14044  (17.9%)  no
  13348  (17.1%)  road
  12561  (16.0%)  council
  11863  (15.2%)  of
  10680  (13.6%)  district
   7439  (9.5%)  and
   7341  (9.4%)  borough
   6416  (8.2%)  tpo
   6168  (7.9%)  land
   5004  (6.4%)  1
   4930  (6.3%)  broadland

--- Most common 2-word phrases in 'tree-preservation-zone' ---
  18789  (24.0%)  tree preservation
  17748  (22.7%)  preservation order
   8978  (11.5%)  district council
   6701  (8.6%)  council tree
   4925  (6.3%)  broadland district
   4845  (6.2%)  the broadland
   4560  (5.8%)  borough of
   3552  (4.5%)  made under
   3332  (4.3%)  order no
   3269  (4.2%)  town and
   3203  (4.1%)  country planning
   3191  (4.1%)  and country
   3106  (4.0%)  the town
   3027  (3.9%)  under the
   2773  (3.5%)  borough council

--- Most common 3-word phrases in 'tree-preservation-zone

In [9]:
# Code-like names for the chosen dataset (reuses `dataset`/`df` set above)
codes = code_like_names(df)
print(f"{len(codes)} rows ({len(codes) / len(df):.1%}) look like bare reference codes in '{dataset}'")
codes[['entity', 'organisation-entity', 'name']]

12050 rows (11.7%) look like bare reference codes in 'tree-preservation-zone'


,entity,organisation-entity,name
9444,19113163,152,W1
9445,19113164,152,W1
9446,19113165,152,W1
9447,19113166,152,W2
9448,19113167,152,W1
...,...,...,...
66425,19178244,205,5001/2025/TPO
66426,19178245,205,5002/2025/TPO
66427,19178246,205,5004/2025/TPO
66428,19178247,205,5006/2025/TPO


In [13]:
# Blank/missing names for the chosen dataset (reuses `dataset`/`df` set above)
blanks = missing_names(df)
print(f"{len(blanks)} rows ({len(blanks) / len(df):.1%}) have a blank/missing name in '{dataset}'")
blanks[['entity', 'organisation-entity', 'name']]

50180 rows (19.1%) have a blank/missing name in 'tree-preservation-zone'


,entity,organisation-entity,name
6552,7002006552,192,NaN
6553,7002006553,192,NaN
6554,7002006554,192,NaN
6555,7002006555,192,NaN
6556,7002006556,192,NaN
...,...,...,...
256556,7002263201,188,NaN
256557,7002263202,188,NaN
256558,7002263203,188,NaN
256565,7002263210,95,NaN


In [14]:
# Swap in whatever phrase looked suspicious above
phrase = "10/12"

matches = df[df['name'].str.contains(phrase, case=False, na=False)]
print(f"{len(matches)} rows in '{dataset}' contain '{phrase}'")
matches[['entity', 'organisation-entity', 'name']].head(20)


11 rows in 'tree-preservation-zone' contain '10/12'


,entity,organisation-entity,name
2283,7002002283,67,South Bucks District Council Tree preservation...
2920,7002002920,67,South Bucks District Council Tree preservation...
2964,7002002964,67,South Bucks District Council Tree preservation...
3175,7002003175,67,South Bucks District Council (No. 20) Tree Pre...
5967,7002005967,67,South Bucks District Council Tree Preservation...
5968,7002005968,67,South Bucks District Council Tree Preservation...
20145,7002020855,48,The London Borough of Barnet 10/12 Thornfield ...
39023,7002042391,234,Located south of 10/12 Chapel Court
39027,7002042395,234,Located east of 10/12 Chapel Court
39031,7002042399,234,Located east of 10/12 Chapel Court


#### Tree

In [ ]:
dataset = "tree" # listed-building-outline, tree-preservation-zone, tree

df = dfs[dataset]
names = df['name'].dropna().astype(str)
for gram_n in range(1, 5):
    print(f"--- Most common {gram_n}-word phrases in '{dataset}' ---")
    for phrase, count in top_ngrams(df, n=gram_n, min_count=5, top=15):
        print(f"  {count:5d}  ({count / len(names):.1%})  {phrase}")
    print()


In [ ]:
# Code-like names for the chosen dataset (reuses `dataset`/`df` set above)
codes = code_like_names(df)
print(f"{len(codes)} rows ({len(codes) / len(df):.1%}) look like bare reference codes in '{dataset}'")
codes[['entity', 'organisation-entity', 'name']]

In [ ]:
# Blank/missing names for the chosen dataset (reuses `dataset`/`df` set above)
blanks = missing_names(df)
print(f"{len(blanks)} rows ({len(blanks) / len(df):.1%}) have a blank/missing name in '{dataset}'")
blanks[['entity', 'organisation-entity', 'name']]


In [ ]:
# Swap in whatever phrase looked suspicious above
phrase = "tree preservation order no"

matches = df[df['name'].str.contains(phrase, case=False, na=False)]
print(f"{len(matches)} rows in '{dataset}' contain '{phrase}'")
matches[['entity', 'organisation-entity', 'name']].head(20)


In [ ]:
# Swap in whatever phrase looked suspicious above
phrase = "tpo"

matches = df[df['name'].str.contains(phrase, case=False, na=False)]
print(f"{len(matches)} rows in '{dataset}' contain '{phrase}'")
matches[['entity', 'organisation-entity', 'name']].head(20)


In [ ]:
# Swap in whatever phrase looked suspicious above
phrase = "Tree Preservation Order 1995"

matches = df[df['name'].str.contains(phrase, case=False, na=False)]
print(f"{len(matches)} rows in '{dataset}' contain '{phrase}'")
matches[['entity', 'organisation-entity', 'name']].head(20)


In [ ]:
# Swap in whatever phrase looked suspicious above
phrase = "Town and Country"

matches = df[df['name'].str.contains(phrase, case=False, na=False)]
print(f"{len(matches)} rows in '{dataset}' contain '{phrase}'")
matches[['entity', 'organisation-entity', 'name']].head(20)

In [ ]:
dupes = duplicate_names(df)
dupes.head(20)